# Part E – LangChain Application: PDF Chatbot

Build an LLM-powered **PDF Question Answering Chatbot** using LangChain.

**Pipeline**: Document Loading → Chunking → Embeddings → Vector DB (FAISS) → Retrieval → LLM Response

**All components run locally** — no cloud APIs required.

In [2]:
pip install langchain-community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

d:\college\VI-sem\DL\lab-task-3\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\college\VI-sem\DL\lab-task-3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Step 1: Load PDF Document

Place your PDF file in the `pdfs/` folder.
Example: `pdfs/sample.pdf`

In [5]:
PDF_PATH = "pdfs/sample.pdf"

assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH}. Place a PDF file there first."

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from {PDF_PATH}")
print(f"First page preview ({len(documents[0].page_content)} chars):")
print(documents[0].page_content[:500])

Loaded 17 pages from pdfs/sample.pdf
First page preview (1566 chars):
UNIT – 1: Introduction to Deep Learning
History of AI →  ML →  DL
Artificial Intelligence (AI - 1950s)
"What if machines could think like humans?"
So they started teaching computers with rules.
Example: “If it has 4 wheels, it’s a car.”
“If it has 2 wings, it’s a plane”
 But the problem was… the world is complicated
Not all cars look the same.
Some planes don’t look like “normal planes.”
 So writing rules for everything became impossible.
Machine Learning (ML 1980s–2000s)
"Instead of writing rul


## Step 2: Text Splitting (Chunking)

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"\nChunk 0 preview:\n{chunks[0].page_content[:300]}")

Split into 68 chunks

Chunk 0 preview:
UNIT – 1: Introduction to Deep Learning
History of AI →  ML →  DL
Artificial Intelligence (AI - 1950s)
"What if machines could think like humans?"
So they started teaching computers with rules.
Example: “If it has 4 wheels, it’s a car.”
“If it has 2 wings, it’s a plane”
 But the problem was… the wor


## Step 3: Create Embeddings + Vector Database (FAISS)

In [7]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": device},
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save for later reuse
vectorstore.save_local("outputs/faiss_index")

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Vector store created with {len(chunks)} vectors")
print("Saved to outputs/faiss_index")

d:\college\VI-sem\DL\lab-task-3\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KARTHIK\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Vector store created with 68 vectors
Saved to outputs/faiss_index


## Step 4: Load Local LLM for Generation

In [8]:
LLM_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_tokenizer.pad_token = llm_tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

text_gen_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

print(f"LLM loaded: {LLM_MODEL} (4-bit quantized)")

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


LLM loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0 (4-bit quantized)


C:\Users\KARTHIK\AppData\Local\Temp\ipykernel_20488\2900784012.py:32: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_gen_pipeline)


## Step 5: Build Retrieval QA Chain

In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Create prompt template
template = """Use the following context to answer the question. 
If you don't know the answer, just say "I don't know based on the provided context."
Keep the answer concise and relevant to the question.

Context:
{context}

Question: {question}

Answer:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])

# Build the chain using LCEL
qa_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)), 
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Retrieval QA chain built successfully!")
print("Components:")
print("  - Document Loader: PyPDFLoader")
print("  - Text Splitter: RecursiveCharacterTextSplitter")
print("  - Embeddings: all-MiniLM-L6-v2")
print("  - Vector Database: FAISS")
print("  - LLM: TinyLlama-1.1B-Chat (4-bit)")
print("  - Chain: LCEL (LangChain Expression Language)")

Retrieval QA chain built successfully!
Components:
  - Document Loader: PyPDFLoader
  - Text Splitter: RecursiveCharacterTextSplitter
  - Embeddings: all-MiniLM-L6-v2
  - Vector Database: FAISS
  - LLM: TinyLlama-1.1B-Chat (4-bit)
  - Chain: LCEL (LangChain Expression Language)


## Step 6: Query the PDF Chatbot

In [10]:
def ask_pdf(question):
    """Ask a question about the loaded PDF."""
    print(f"\nQ: {question}")
    print("-" * 50)
    
    # Get answer from chain
    answer = qa_chain.invoke(question)
    
    # Get source documents separately
    source_docs = retriever.invoke(question)
    
    print(f"A: {answer}\n")
    print("Source chunks used:")
    for i, doc in enumerate(source_docs):
        page = doc.metadata.get("page", "?")
        print(f"  [{i+1}] Page {page}: {doc.page_content[:100]}...")
    print("=" * 60)
    return answer


# Example queries — modify these based on your PDF content
sample_questions = [
    "What is the main topic of this document?",
    "Can you summarize the key points?",
    "What are the conclusions mentioned in this document?",
]

print("=" * 60)
print("PDF CHATBOT - Question Answering Results")
print("=" * 60)

for question in sample_questions:
    ask_pdf(question)

PDF CHATBOT - Question Answering Results

Q: What is the main topic of this document?
--------------------------------------------------
A: Use the following context to answer the question. 
If you don't know the answer, just say "I don't know based on the provided context."
Keep the answer concise and relevant to the question.

Context:
UNIT – 1: Introduction to Deep Learning
History of AI →  ML →  DL
Artificial Intelligence (AI - 1950s)
"What if machines could think like humans?"
So they started teaching computers with rules.
Example: “If it has 4 wheels, it’s a car.”
“If it has 2 wings, it’s a plane”
 But the problem was… the world is complicated
Not all cars look the same.
Some planes don’t look like “normal planes.”
 So writing rules for everything became impossible.
Machine Learning (ML 1980s–2000s)

They read one word at a time →  too slow.
They forget long context →  bad at long sentences.
Hard to parallelize →  training took ages.
That’s where Transformers came in (2017, “Atte

## Step 7: Interactive Chat Mode (Optional)

Uncomment the cell below to interactively chat with your PDF.

In [14]:
print("PDF Chatbot - Interactive Mode")
print("Type 'quit' to exit\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    ask_pdf(user_input)

PDF Chatbot - Interactive Mode
Type 'quit' to exit


Q: what is the main topic in the pdf
--------------------------------------------------
A: Use the following context to answer the question. 
If you don't know the answer, just say "I don't know based on the provided context."
Keep the answer concise and relevant to the question.

Context:
They read one word at a time →  too slow.
They forget long context →  bad at long sentences.
Hard to parallelize →  training took ages.
That’s where Transformers came in (2017, “Attention is All You Need”).
Core Idea: Attention Mechanism
Imagine you’re reading this sentence:
“The dog chased the ball because it was rolling fast.”
Question: What does “it” refer to? →  the ball.
Attention helps the model focus on the important word (ball) instead of treating all words equally.

It assigns weights (importance scores) to words.
So, when the model sees “it”, attention tells it: “Look at ball, not dog.”
Transformer Architecture
Input Embedding – Words con

KeyboardInterrupt: 

## Pipeline Summary

```
PDF Document
    |
    v
PyPDFLoader (Document Loading)
    |
    v
RecursiveCharacterTextSplitter (Chunking: 500 chars, 50 overlap)
    |
    v
all-MiniLM-L6-v2 (Sentence Embeddings)
    |
    v
FAISS Vector Database (Similarity Search)
    |
    v
Top-3 Relevant Chunks Retrieved
    |
    v
TinyLlama-1.1B-Chat (Local LLM, 4-bit quantized)
    |
    v
Generated Answer
```